# XAI for neural networks and its evaluation

## Practical 1 - part 1 - PyTorch tensors and Convolutional Neural Networks (CNNs)

This notebook serves as a recap of the main functionalities of PyTorch for Deep Learning. The outcome of today's practical is to go over the main functionalities of PyTorch, including the basics of tensor calculus, usage of CPU vs GPU, automatic gradient computation, and training a CNN from scratch.

In [1]:
import torch
print(torch.__version__)

2.15.0+cu132


We first set up PyTorch to work in a device-agnostic way.

For its computations, we want to use a GPU if it is available; otherwise we stay on a CPU.
Additionally, PyTorch offers sypport for Apple Silicon Macs (`mps` device).

There are additional notes on possible devices:

1. You can use AMD GPUs in the same ways as Nvidia GPUs by installing a specialized ROCm PyTorch version (e.g, `pip install torch torchvision --index-url https://download.pytorch.org/whl/rocm6.2`)
2. You can run PyTorch on TPUs by using the `torch-tpu` package (`pip install torch-tpu --extra-index-url https://download.pytorch.org/whl/cpu`). You need to specify the usage of a TPU device in this case (`device = torch_xla.device()`).

In [2]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

Using device: cuda


### Tensors

At the basis of PyTorch, is a tensor computation engine that allows for fast and readable computations between n-dimensional objects on CPU or GPU.

Tensors are multi-dimensional abstractions of vectors or matrices.

1d tensor = vector
2d tensor = matrix
3d tensor = "cubic" matrix, vector of matrices
etc

![](https://www.avni.sh/posts/linear-algebra/tensors/tensors-cover.webp)

Learn more here: https://www.avni.sh/posts/linear-algebra/tensors/

In [3]:
# this is a way to create a tensor with known entries
x = torch.tensor(
    [[1, 2, 3],[4, 5, 6],[7, 8, 9]],
    device=device # notice that we can create the tensor on a specific device
)
print(x)

# shape or size() allows us to print the size and number of dimensions of the tensor
print(x.shape)

tensor([[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]], device='cuda:0')
torch.Size([3, 3])


In [4]:
# we can quickly shift the tensor to CPU using .to
x.to("cpu")

tensor([[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]])

In [6]:
# we can also create tensors with random entries and arbitrary dimensions (e.g., 2x3x4)
y = torch.rand(2,3,4, device = device)
print(y)

tensor([[[0.3144, 0.1820, 0.1635, 0.7126],
         [0.5224, 0.4188, 0.0730, 0.2437],
         [0.2884, 0.2545, 0.9049, 0.6799]],

        [[0.3050, 0.2936, 0.6776, 0.3230],
         [0.2711, 0.0634, 0.6090, 0.4895],
         [0.0066, 0.9542, 0.2603, 0.5691]]], device='cuda:0')


In [ ]:
# convert the 3d tensor into a matrix
y.reshape(2, 12)

### Tensor calculations and autograd

PyTorch natively supports all ranges of tensor computations.
When operations are applied to tensors, PyTorch creates a computation graph that keeps track of all the computations that have been applied to it.

PyTorch can always recall the graph and compute the gradients on the root and intermediate variables on-demand.


Let us reproduce this example with scalars:

![|1](https://simple-english-machine-learning.readthedocs.io/en/latest/_images/colah-blog-basic-computational-graph-calculate.png)

In [8]:
a = torch.tensor(2.0, requires_grad=True) # requires_grad needed to tell PyTorch to keep track of gradients
b = torch.tensor(1.0, requires_grad=True)

c = a + b
d = b + 1
e = c * d

print("e =",e)

e.backward() # compute gradients

print("de/da", a.grad)
print("de/db", b.grad)


e = tensor(6., grad_fn=<MulBackward0>)
de/da tensor(2.)
de/db tensor(5.)


This works also with any tensor dimension, as long as the backward step is initiated on a scalar tensor.

This procedure is at the core of the (Stochastic) Gradient Descent algorithm, vital for training a neural network.

In [7]:
a = torch.tensor([1.,2,3], requires_grad=True)
b = torch.tensor([[4.,5,6],[7,8,9]], requires_grad=True)

c = a + b
print("c =", c)
d = b + 1
print("d =", d)
e = c * d
print("e =",e)

f = e.sum() # sum all elements in matrix
print("f =", f)

f.backward() # compute gradients

print("df/da", a.grad)
print("df/db", b.grad)


c = tensor([[ 5.,  7.,  9.],
        [ 8., 10., 12.]], grad_fn=<AddBackward0>)
d = tensor([[ 5.,  6.,  7.],
        [ 8.,  9., 10.]], grad_fn=<AddBackward0>)
e = tensor([[ 25.,  42.,  63.],
        [ 64.,  90., 120.]], grad_fn=<MulBackward0>)
f = tensor(404., grad_fn=<SumBackward0>)
df/da tensor([13., 15., 17.])
df/db tensor([[10., 13., 16.],
        [16., 19., 22.]])


We can also disable the creation of the computation graph with the inference mode.
Despite the `requires_grad` flag, torch will not compute gradients, which greatly speeds up execution (useful when the model has been fully trained).

In [ ]:
# reset gradients
a.grad = None
b.grad = None

with torch.inference_mode():
    c = a + b
    d = b + 1
    e = c * d
    f = e.sum()
    print("f =", f)

f.backward() # raise error because inference mode, which desactivate grad (and therefore f does not require grad)

f = tensor(6.)


RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

### Creating and training a (C)NN

PyTorch NNs are coded using Object-Oriented Programming principles.

A model contains the parametric structure (e.g., the layers in a NN) and the logic that allows to generate prediction given inputs.

A model is a class inheriting from torch.nn.module and needs to implement two methods:
1. A constructor (which sets up the computation layers)
2. A forward pass (which defines how the input passes through the model and the output is produced)

There are optional methods which can be implemented, e.g., _init_weights, which defines custom initialization for the parameters.

This is a sample code for a 2-layer CNN:
* Layer 1: 3x3 kernel, 32 output channels
* Layer 2: 3x3 kernel, 64 output channels
* Adaptive Average Pooling: returns a feature map of size 64x4x4
* Classifier: flattens the pooled features and applies a single linear layer with 10 output neurons

The CNN returns a Tensor of size n x 10, where n is the number of input data points. This represents *logits*, proportional to the probability of assignment to the given class. An example of output for 2 input data points:

|Class 0|Class 1|Class 2|Class 3|Class 4|Class 5|Class 6|Class 7|Class 8|Class 9|
|---|---|---|---|---|---|---|---|---|---|
|0.5|0.0|1.1|2.5|-1.3|-6.1|4.5|0.1|3.7|-2.4|
|0.0|3.7|-2.4|-10.2|3.1|1.1|-1.4|0.1|4.2|-1.1

**Q:** *What are the classes assigned to these two inputs? How can we compute them?*


In [ ]:
from torch import nn

class CompactCNN(nn.Module):
    """Simple, clean 3-layer CNN architecture for single-channel/RGB classification."""

    def __init__(self, in_channels: int = 1, num_classes: int = 10):
        super().__init__()

        # Conv Block 1: Feature Extraction
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),  # BatchNorm reduces need for Conv bias
            nn.SiLU(),           # Modern activation (Swish)
            nn.MaxPool2d(kernel_size=2, stride=2),  # H, W -> H/2, W/2

            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.SiLU(),
            nn.AdaptiveAvgPool2d((4, 4))  # Fixed output feature map size regardless of input spatial dims
        )

        # Classification Head
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(p=0.2),
            nn.Linear(64 * 4 * 4, num_classes)
        )

        # Custom Weight Initialization
        self._init_weights()

    def _init_weights(self) -> None:
        """Kaiming/He initialization suited for SiLU/ReLU activations."""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.feature_extractor(x)
        logits = self.classifier(x)
        return logits

The training loop has to be coded by hand.
There are multiple philosophical lines as to how to code the loop.
We can create a function to train a single loop. The entities involved in this process are:

1. The model
2. The dataloader (takes a dataset and manages the unpacking in batches, applying preprocessing and data augmentation in the process)
3. The loss function (a.k.a. criterion)
4. The optimizer
5. The scaler (manages precision of computation on GPU)
6. The device on which compute happens

The trainer
1. Sends the data to the device
2. Computes the outputs by applying the model to the inputs (notice, output is n x 10)
3. Computes the predicted classes by selecting the row-wise argmax
4. Computes the loss using an adaptive FP16/64 precision (for optimal speedup)
5. Calculates the gradients using backward()
6. Updates the parameters based on the gradients
7. Keeps track of the correct predictions for each batch
8. Returns the average loss and accuracy for the epoch

In [ ]:
from torch.utils.data import DataLoader
from torch import optim
import torchmetrics
import torch


def train_one_epoch(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    scaler: torch.amp.GradScaler,
    device: torch.device
) -> tuple[float, float]:
    '''
    Execute one training epoch using torchmetrics for both accuracy and loss tracking.
    Saves a checkpoint of the model and optimizer states at the end.
    '''
    model.train()

    # Initialize metrics from torchmetrics
    accuracy_metric = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
    loss_metric = torchmetrics.MeanMetric().to(device)

    for batch_idx, (inputs, targets) in enumerate(dataloader):
        inputs, targets = inputs.to(device, non_blocking=True), targets.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        # Automatic Mixed Precision (AMP) context
        with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            predicted_classes = outputs.argmax(dim=1)

        # Mixed Precision Scaled Backward & Step
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Update both metrics
        accuracy_metric(predicted_classes, targets)
        loss_metric(loss)

    # Compute final epoch metrics and reset
    epoch_loss = loss_metric.compute().item()
    epoch_acc = accuracy_metric.compute().item()

    loss_metric.reset()
    accuracy_metric.reset()

    # Save checkpoint state for both model (i.e., parameters) and optimizer
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict()
    }
    torch.save(checkpoint, "checkpoint_epoch.pt")

    return epoch_loss, epoch_acc

### The execution

For time constraints, we showcase the training of the custom CNN on MNIST.
We just show the execution for 2 batches. Notice that training time should possibly be higher to attain competitive results.

We first show the data loaders creation: for this, we need to download the data online and pass it to the dataloader.
The data also needs preoprocessing steps, which for us are simply:
1. Transformation of the image to tensor
2. Standardization so the data has mean 0 and standard deviation 1

In [ ]:
from torchvision import transforms
from torchvision import datasets

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # Standard MNIST mean and std - these data need to be determined beforehand
])

# Download and load training/test datasets
train_dataset = datasets.MNIST(
    root="./data", train=True, download=True, transform=transform
)
test_dataset = datasets.MNIST(
    root="./data", train=False, download=True, transform=transform
)

# DataLoader handles batching, shuffling, and multi-process data loading efficiently
train_loader = DataLoader(
    train_dataset, batch_size=64, shuffle=True, pin_memory=True, num_workers=2
)
test_loader = DataLoader(
    test_dataset, batch_size=1000, shuffle=False, pin_memory=True
)

We then create the model, the optimizer, define the loss function and the scaler, then pack all these together in the 2 training epochs

In [ ]:
num_epochs = 2

model = CompactCNN(in_channels=1, num_classes=10).to(device)

optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

# Run one quick training epoch to verify pipeline
for epoch in range(num_epochs):
  loss, acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, device)
  print(f"Epoch {epoch+1} Loss: {loss:.4f} | Accuracy: {acc * 100:.2f}%")

### Evaluating the model

We can now evaluate the model on the test set with a loop similar to training

In [ ]:
import torch
import torchmetrics
from torch import nn
from torch.utils.data import DataLoader

@torch.inference_mode()
def evaluate_model(
    model: nn.Module,
    dataloader: DataLoader,
    criterion: nn.Module,
    device: torch.device
) -> tuple[float, float]:
    '''
    Evaluate the model on a test/validation dataloader.
    Returns the average loss and accuracy using torchmetrics.
    '''
    model.eval()

    # Initialize metrics and move to device
    accuracy_metric = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)
    loss_metric = torchmetrics.MeanMetric().to(device)

    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device, non_blocking=True), targets.to(device, non_blocking=True)

        # Compute outputs and loss
        outputs = model(inputs)
        loss = criterion(outputs, targets) # notice: not necessarily needed for evaluation pipeline
        predicted_classes = outputs.argmax(dim=1)

        # Update metrics
        accuracy_metric(predicted_classes, targets)
        loss_metric(loss)

    # Compute final metrics and reset state
    eval_loss = loss_metric.compute().item()
    eval_acc = accuracy_metric.compute().item()

    loss_metric.reset()
    accuracy_metric.reset()

    return eval_loss, eval_acc

In [ ]:
eval_loss, eval_acc = evaluate_model(model, test_loader, criterion, device)
print(f"Test Loss: {eval_loss:.4f} | Test Accuracy: {eval_acc * 100:.2f}%")

### Finetuning models

A far more time-saving approach is to finetune existing models.

We will show an example with a ResNet18 pre-trained on ImageNet and we will fine-tune it for a few epochs on CIFAR10.

1. We can finetune by using the model as a feature extractor (take only the `feature_extractor` component, remove the classifier) and by training a small classifier on top of it, or even another model, like an SVM.
2. We can "freeze" some of the initial layer and train only some layers at the end of the model.
3. We can retrain the whole model for a few epochs, but using an optimizer with a small learning rate.

In [ ]:
# OPTION 1 - USE FEATURE EXTRACTOR ONLY

from torchvision import datasets, transforms, models

# Prepare CIFAR-10 datasets with ResNet18 normalization and resizing
cifar_transform_train = transforms.Compose([
    transforms.Resize((224, 224)),  # ResNet18 optimal input size
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

cifar_transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

cifar_train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=cifar_transform_train)
cifar_test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=cifar_transform_test)

cifar_train_loader = DataLoader(cifar_train_dataset, batch_size=64, shuffle=True, pin_memory=True, num_workers=2)
cifar_test_loader = DataLoader(cifar_test_dataset, batch_size=64, shuffle=False, pin_memory=True, num_workers=2)

# Load Pretrained ResNet18
weights = models.ResNet18_Weights.DEFAULT
resnet = models.resnet18(weights=weights)

# Freeze all feature extraction layers
for param in resnet.parameters():
    param.requires_grad = False

# Replace the final fully connected layer for CIFAR-10 classification
num_features = resnet.fc.in_features
resnet.fc = nn.Sequential(
    nn.Linear(num_features, 256),
    nn.SiLU(),
    nn.Dropout(p=0.3),
    nn.Linear(256, 10)
)

resnet = resnet.to(device)

# Define optimizer (only training the parameters of the new fc head), criterion and scaler
optimizer_resnet = optim.AdamW(resnet.fc.parameters(), lr=1e-3, weight_decay=1e-4)
criterion_resnet = nn.CrossEntropyLoss()
scaler_resnet = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

# 6. Fine-tune using our existing helper functions
num_epochs_resnet = 2
print("Starting fine-tuning...")
for epoch in range(num_epochs_resnet):
    train_loss, train_acc = train_one_epoch_with_tracked_loss(
        resnet, cifar_train_loader, criterion_resnet, optimizer_resnet, scaler_resnet, device
    )
    print(f"Epoch {epoch+1}/{num_epochs_resnet} | Train Loss: {train_loss:.4f} | Train Accuracy: {train_acc * 100:.2f}%")

# Evaluate on CIFAR-10 test set
test_loss, test_acc = evaluate_model(resnet, cifar_test_loader, criterion_resnet, device)
print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc * 100:.2f}%")

Note: option 2 follows trivially - instead of removing the classifier, you just selectiveley freeze some layers with the same code you have seen here

```python
# Freeze all feature extraction layers
for param in resnet.parameters():
    param.requires_grad = False
```

For option 3, you can selectively take the code needed from the cell above.
Don't freeze any layers, just train with a small learning rate (e.g., start with 1e-4 or 1e-5).

### Training on your own dataset for image data

If the dataset is not available on the internet and you have it locally, the best option is to organize the data in your filesystem using this hierarchical structure:

```
data
  |
  - class_1
      |
      - image_1_class_1.jpg
      - image_2_class_1.jpg
      ...
  - class_2
      |
      - image_1_class_2.jpg
      - image_2_class_2.jpg
      ...
```

You can seamlessly re-use the `ImageFolder` dataset from torchvision to import this data.